In [ ]:
# MATLAB section 1/17 for PPSimExample: General Point Process Simulation

# % General Point Process Simulation
# In this demo, we show how sample-paths of a point process (PP) can be
# generated from specification of its conditional intensity function (CIF).
# We then use the generated PP data to validate the outputs of the Neural
# Spike Analysis Toolbox.

# Python translation bootstrap for this helpfile.

# Topic: PPSimExample
# Execution group: smoke
# Workflow family: network
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/PPSimExample.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PPSimExample"
FAMILY = "network"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PPSimExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PPSimExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PPSimExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PPSimExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 6

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)


In [ ]:
# MATLAB section 2/17 for PPSimExample: Point Process Sample Path Generation

# % Point Process Sample Path Generation
# That both the stimulus effect and ensemble effects can be made into
# multi-input/multi-output transfer functions to account for more than 1
# stimulus effect or multiple neighboring neuron effects. To do this,
# simply define $$E$ or $$S$ to be a row vector of LTI transfer functions.
# Make sure than the number of dimensions of the input matches the number
# of transfer functions specified in the row vector.
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/17 for PPSimExample: Section

# %
#
# <<PPSimExample-BlockDiagram.png>>
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/17 for PPSimExample: Section

# %
# This block diagram specifies a conditional intensity function of the form
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/17 for PPSimExample: Section

# %
#
# $$\lambda_{i} \cdot \Delta = exp(\mu_{i} + H*\Delta N_{i}[n] + S*u_{stim}[n] + E*\Delta N_{k}[n])/(1+exp(\mu_{i} + H*\Delta N_{i}[n] + S*u_{stim}[n] + E*\Delta N_{k}[n]))$$
#
#
#
#
# MATLAB: close all;
# MATLAB: Ts=.001;            %Sample Time
# MATLAB: tMin=0; tMax=50;    %Simulation duration
# MATLAB: t=tMin:Ts:tMax;
#
# MATLAB: mu=-3;              %Baseline firing rate of the neurons being modeled
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/17 for PPSimExample: History Effect

# % History Effect
#
# $$1*h[n]=-1*\Delta N[n-1]-2*\Delta N[n-2] -4*\Delta N[n-3]$$
#
# MATLAB: H=tf([-1 -2 -4],[1],Ts,'Variable','z^-1');
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 6

_ = section_index


In [ ]:
# MATLAB section 7/17 for PPSimExample: Stimulus Effect

# % Stimulus Effect
#
# $$1*s[n]=1*u_{stim}[n]$$
#
# MATLAB: S=tf([1],1,Ts,'Variable','z^-1');
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 7

_ = section_index


In [ ]:
# MATLAB section 8/17 for PPSimExample: Ensemble Effect

# % Ensemble Effect
#
# $$1*e[n]=0*\Delta N_{k}[n]$$
#
# MATLAB: E=tf([0],1,Ts,'Variable','z^-1');
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 8

_ = section_index


In [ ]:
# MATLAB section 9/17 for PPSimExample: Section

# %
# MATLAB: f=1;                    %Stimulus frequency
# MATLAB: u = sin(2*pi*f*t)';       %Make this neuron modulated by a sine wave
# MATLAB: e = zeros(length(t),1);   %No Ensemble input
#
# MATLAB: stim=Covariate(t',u,'Stimulus','time','s','Voltage',{'sin'});
# MATLAB: ens =Covariate(t',e,'Ensemble','time','s','Spikes',{'n1'});
# MATLAB: numRealizations = 5;    %Number of sample paths to generate
# MATLAB: fitType = 'binomial';
# MATLAB: sC=CIF.simulateCIF(mu,H,S,E,stim,ens,numRealizations,fitType);
# MATLAB: figure;
# MATLAB: subplot(2,1,1); sC.plot;    v=axis; axis([0 tMax/10 v(3) v(4)]);
# MATLAB: subplot(2,1,2); stim.plot;  v=axis; axis([0 tMax/10 v(3) v(4)]);
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=9, matlab_line_number=66, matlab_snippet="figure;")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPSimExample-BlockDiagram.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 9, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 9

_ = section_index


In [ ]:
# MATLAB section 10/17 for PPSimExample: GLM Model Fitting Setup

# % GLM Model Fitting Setup
# In this section, we create the appropriate structures to fit several GLM
# models to the data generated above.
#
# Create a constant covariate representing the mean firing rate $$\mu_{i}$
# MATLAB: baseline=Covariate(t',ones(length(t),1),'Baseline','time','s','',{'mu'});
#
#
# MATLAB: spikeColl = sC;               %Use the generated data as our collection of spikes
# MATLAB: cc=CovColl({stim,baseline});  %Use stimulation and baseline as possible covariates
# MATLAB: trial = Trial(spikeColl,cc); sampleRate = 1/Ts; %Create trial
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 10

_ = section_index


In [ ]:
# MATLAB section 11/17 for PPSimExample: GLM Model Fitting and Results

# % GLM Model Fitting and Results
# MATLAB: clear c;
# MATLAB: selfHist = [0:0.001:0.003]; %We know the history effect goes back 3 lag orders
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 11

_ = section_index


In [ ]:
# MATLAB section 12/17 for PPSimExample: Section

# %
# Fit only a mean firing rate
# MATLAB: c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);
# MATLAB: c{1}.setName('Baseline');
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 12

_ = section_index


In [ ]:
# MATLAB section 13/17 for PPSimExample: Section

# %
# Fit a mean firing rate + the stimulus term
# MATLAB: c{2} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,[],[]);
# MATLAB: c{2}.setName('Stim');
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 13

_ = section_index


In [ ]:
# MATLAB section 14/17 for PPSimExample: Section

# %
# Fit a mean firing rate, self-history, and stimulus --- Same as true model
# MATLAB: c{3} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,selfHist,[]);
# MATLAB: c{3}.setName('Stim+Hist');
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 14

_ = section_index


In [ ]:
# MATLAB section 15/17 for PPSimExample: Section

# %
# Place all configurations together and run analysis for each neuron
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: if(strcmp(fitType,'binomial'))
# MATLAB:     Algorithm = 'BNLRCG';   % BNLRCG - faster Truncated, L-2 Regularized,
# Binomial Logistic Regression with Conjugate
# Gradient Solver by Demba Ba (demba@mit.edu).
# MATLAB: else
# MATLAB:     Algorithm = 'GLM';      % Standard Matlab GLM (Can be used for binomial or
# or Poisson CIFs
# MATLAB: end
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0,Algorithm);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 15

_ = section_index


In [ ]:
# MATLAB section 16/17 for PPSimExample: Results for sample neuron

# % Results for sample neuron
# MATLAB: results{1}.plotResults;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 16

_ = section_index


In [ ]:
# MATLAB section 17/17 for PPSimExample: Results for across all sample paths

# % Results for across all sample paths
#
# MATLAB: Summary = FitResSummary(results);
# MATLAB: Summary.plotSummary;

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: <synthetic MATLAB figure event #2 for PPSimExample>
fig = FIGURE_TRACKER.new_figure(section_index=17, matlab_line_number=118, matlab_snippet="<synthetic MATLAB figure event #2 for PPSimExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPSimExample.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 17, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #3 for PPSimExample>
fig = FIGURE_TRACKER.new_figure(section_index=17, matlab_line_number=118, matlab_snippet="<synthetic MATLAB figure event #3 for PPSimExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPSimExample_01.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 17, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #4 for PPSimExample>
fig = FIGURE_TRACKER.new_figure(section_index=17, matlab_line_number=118, matlab_snippet="<synthetic MATLAB figure event #4 for PPSimExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPSimExample_02.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 17, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #5 for PPSimExample>
fig = FIGURE_TRACKER.new_figure(section_index=17, matlab_line_number=118, matlab_snippet="<synthetic MATLAB figure event #5 for PPSimExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPSimExample_03.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 17, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #6 for PPSimExample>
fig = FIGURE_TRACKER.new_figure(section_index=17, matlab_line_number=118, matlab_snippet="<synthetic MATLAB figure event #6 for PPSimExample>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPSimExample_04.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=6 + 17, title=f"{TOPIC} Figure 006")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "Ts=.001;            %Sample Time",
    "tMin=0; tMax=50;    %Simulation duration",
    "t=tMin:Ts:tMax;",
    "mu=-3;              %Baseline firing rate of the neurons being modeled",
    "H=tf([-1 -2 -4],[1],Ts,'Variable','z^-1');",
    "S=tf([1],1,Ts,'Variable','z^-1');",
    "E=tf([0],1,Ts,'Variable','z^-1');",
    "f=1;                    %Stimulus frequency",
    "u = sin(2*pi*f*t)';       %Make this neuron modulated by a sine wave",
    "e = zeros(length(t),1);   %No Ensemble input",
    "stim=Covariate(t',u,'Stimulus','time','s','Voltage',{'sin'});",
    "ens =Covariate(t',e,'Ensemble','time','s','Spikes',{'n1'});",
    "numRealizations = 5;    %Number of sample paths to generate",
    "fitType = 'binomial';",
    "sC=CIF.simulateCIF(mu,H,S,E,stim,ens,numRealizations,fitType);",
    "figure;",
    "subplot(2,1,1); sC.plot;    v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "subplot(2,1,2); stim.plot;  v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "baseline=Covariate(t',ones(length(t),1),'Baseline','time','s','',{'mu'});",
    "spikeColl = sC;               %Use the generated data as our collection of spikes",
    "cc=CovColl({stim,baseline});  %Use stimulation and baseline as possible covariates",
    "trial = Trial(spikeColl,cc); sampleRate = 1/Ts; %Create trial",
    "clear c;",
    "selfHist = [0:0.001:0.003]; %We know the history effect goes back 3 lag orders",
    "c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,[],[]);",
    "c{2}.setName('Stim');",
    "c{3} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,selfHist,[]);",
    "c{3}.setName('Stim+Hist');",
    "cfgColl= ConfigColl(c);",
    "if(strcmp(fitType,'binomial'))",
    "Algorithm = 'BNLRCG';   % BNLRCG - faster Truncated, L-2 Regularized,",
    "else",
    "Algorithm = 'GLM';      % Standard Matlab GLM (Can be used for binomial or",
    "end",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0,Algorithm);",
    "results{1}.plotResults;",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for PPSimExample.")

# PPSimExample: fixture-backed Poisson GLM simulation and parity checks.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/PPSimExample_gold.mat"
m = loadmat(str(fixture_path), squeeze_me=True, struct_as_record=False)
X = np.asarray(m["X"], dtype=float).reshape(-1, 1)
y = np.asarray(m["y"], dtype=float).reshape(-1)
dt = float(np.asarray(m["dt"], dtype=float).reshape(-1)[0])
expected_rate = np.asarray(m["expected_rate"], dtype=float).reshape(-1)
b = np.asarray(m["b"], dtype=float).reshape(-1)
fit = Analysis.fit_glm(X=X, y=y, fit_type="poisson", dt=dt)
pred_rate = np.asarray(fit.predict(X), dtype=float).reshape(-1)
rel_err = float(np.mean(np.abs(pred_rate - expected_rate) / np.maximum(expected_rate, 1e-12)))
intercept_abs_error = float(abs(fit.intercept - b[0]))
coeff_abs_error = float(abs(fit.coefficients[0] - b[1]))
assert rel_err <= 0.25 and intercept_abs_error <= 0.25 and coeff_abs_error <= 0.25
time = np.arange(X.shape[0]) * dt
stim = X.reshape(-1)
spike_idx = np.where(y > 0)[0]

fig, axes = plt.subplots(3, 1, figsize=(10.2, 7.4), sharex=False)
axes[0].plot(time, stim, "k", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: driving stimulus")
axes[0].set_ylabel("stim")
axes[1].vlines(time[spike_idx], 0.6, 1.4, color="black", linewidth=0.35)
axes[1].set_title("Point-process sample path")
axes[1].set_ylabel("trial #1")
axes[2].plot(time, expected_rate, color="tab:green", linewidth=1.0, linestyle="--", label="MATLAB gold")
axes[2].plot(time, pred_rate, color="tab:red", linewidth=1.0, label="Python fit")
axes[2].plot(time, y / max(dt, 1e-12), color="0.7", linewidth=0.3, alpha=0.5, label="counts/dt")
axes[2].set_xlabel("time [s]")
axes[2].set_ylabel("Hz")
axes[2].set_title("Conditional intensity fit")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

CHECKPOINT_METRICS = {
    "mean_simulated_rate": float(np.mean(pred_rate)),
    "relative_rate_error": rel_err,
    "intercept_abs_error": intercept_abs_error,
    "coeff_abs_error": coeff_abs_error,
}
CHECKPOINT_LIMITS = {
    "mean_simulated_rate": (0.1, 500.0),
    "relative_rate_error": (0.0, 0.25),
    "intercept_abs_error": (0.0, 0.25),
    "coeff_abs_error": (0.0, 0.25),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
